# Library

In [15]:
import pandas as pd
import numpy as np
import os

# 데이터가 있는 폴더 경로 (본인 환경에 맞게 수정)
INPUT_DIR = '../provided'
OUTPUT_DIR = './' # preprocessed 폴더 위치

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Base Data Preprocessing

## base_seeds_features

In [16]:
# 원본 시드 데이터 로드
m_seeds = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneySeeds.csv'))
w_seeds = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneySeeds.csv'))

# 문자열에서 숫자 부분만 추출하는 함수
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)
w_seeds['SeedNum'] = w_seeds['Seed'].apply(parse_seed)

# 필요한 컬럼만 추출하여 저장
base_seeds_m = m_seeds[['Season', 'TeamID', 'SeedNum']]
base_seeds_w = w_seeds[['Season', 'TeamID', 'SeedNum']]

base_seeds_m.to_csv(os.path.join(OUTPUT_DIR, 'base_seeds_features_M.csv'), index=False)
base_seeds_w.to_csv(os.path.join(OUTPUT_DIR, 'base_seeds_features_W.csv'), index=False)
print("A 파트 저장 완료: base_seeds_features_M.csv / base_seeds_features_W.csv")


A 파트 저장 완료: base_seeds_features_M.csv / base_seeds_features_W.csv


## base_season_stats_features

In [17]:
# 원본 정규시즌 성적 로드
m_reg = pd.read_csv(os.path.join(INPUT_DIR, 'MRegularSeasonCompactResults.csv'))
w_reg = pd.read_csv(os.path.join(INPUT_DIR, 'WRegularSeasonCompactResults.csv'))

def build_season_stats(reg_df):
    # 이긴 팀 관점 데이터 추가
    w = reg_df[['Season', 'WTeamID', 'WScore', 'LScore']].copy()
    w.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    w['Win'] = 1
    
    # 진 팀 관점 데이터 추가
    l = reg_df[['Season', 'LTeamID', 'LScore', 'WScore']].copy()
    l.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    l['Win'] = 0
    
    # 병합
    df = pd.concat([w, l], ignore_index=True)
    df['Margin'] = df['ScoreFor'] - df['ScoreAgainst']
    
    # 그룹화 및 통계 수치(Features) 집계
    stats = df.groupby(['Season', 'TeamID']).agg(
        Games=('Win', 'count'),
        Wins=('Win', 'sum'),
        AvgScore=('ScoreFor', 'mean'),
        AvgMargin=('Margin', 'mean'),
    ).reset_index()
    
    stats['WinRate'] = stats['Wins'] / stats['Games']
    
    # 문서 명세에 맞춰 컬럼 순서 정리
    return stats[['Season', 'TeamID', 'Games', 'Wins', 'WinRate', 'AvgScore', 'AvgMargin']]

base_season_stats_m = build_season_stats(m_reg)
base_season_stats_w = build_season_stats(w_reg)

base_season_stats_m.to_csv(os.path.join(OUTPUT_DIR, 'base_season_stats_features_M.csv'), index=False)
base_season_stats_w.to_csv(os.path.join(OUTPUT_DIR, 'base_season_stats_features_W.csv'), index=False)
print("B 파트 저장 완료: base_season_stats_features_M.csv / base_season_stats_features_W.csv")


B 파트 저장 완료: base_season_stats_features_M.csv / base_season_stats_features_W.csv


## base_massey_features

In [18]:
# 매시 랭킹 데이터(남자부 전용) 로드
massey = pd.read_csv(os.path.join(INPUT_DIR, 'MMasseyOrdinals.csv'))

# MOR 시스템 한정, 시즌/팀별 최신 순위(last) 추출
base_massey = (
    massey[massey['SystemName'] == 'MOR']
    .sort_values(['Season', 'RankingDayNum'])
    .groupby(['Season', 'TeamID']).last()
    .reset_index()
    [['Season', 'TeamID', 'OrdinalRank']]
    .rename(columns={'OrdinalRank': 'MOR'}) # 지정한 변수명으로 변경
)

base_massey.to_csv(os.path.join(OUTPUT_DIR, 'base_massey_features.csv'), index=False)
print("C 파트 저장 완료: base_massey_features.csv")


C 파트 저장 완료: base_massey_features.csv


## base_matchup_features

In [19]:
# 원본 토너먼트 결과 로드
m_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneyCompactResults.csv'))
w_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneyCompactResults.csv'))

def get_base_feats(season, team_id, stats, seed_df, mor_df=None):
    s_row = stats[(stats.Season == season) & (stats.TeamID == team_id)]
    e_row = seed_df[(seed_df.Season == season) & (seed_df.TeamID == team_id)]
    win_rate = s_row['WinRate'].values[0] if len(s_row) else np.nan
    avg_margin = s_row['AvgMargin'].values[0] if len(s_row) else np.nan
    avg_score = s_row['AvgScore'].values[0] if len(s_row) else np.nan
    seed_num = e_row['SeedNum'].values[0] if len(e_row) else np.nan
    mor = np.nan
    if mor_df is not None:
        m_row = mor_df[(mor_df.Season == season) & (mor_df.TeamID == team_id)]
        mor = m_row['MOR'].values[0] if len(m_row) else np.nan
    return win_rate, avg_margin, avg_score, seed_num, mor

def build_base_matchup_df(tourney, stats, seed_df, mor_df=None):
    rows = []
    for _, r in tourney.iterrows():
        s = r['Season']
        t1, t2 = sorted([r['WTeamID'], r['LTeamID']])
        label = 1 if r['WTeamID'] == t1 else 0
        f1 = get_base_feats(s, t1, stats, seed_df, mor_df)
        f2 = get_base_feats(s, t2, stats, seed_df, mor_df)
        rows.append(dict(
            Season=s, T1=t1, T2=t2,
            T1_WinRate=f1[0], T2_WinRate=f2[0],
            T1_Margin=f1[1], T2_Margin=f2[1],
            T1_Score=f1[2], T2_Score=f2[2],
            T1_Seed=f1[3], T2_Seed=f2[3],
            T1_MOR=f1[4], T2_MOR=f2[4],
            Label=label
        ))
    df = pd.DataFrame(rows)
    df['SeedDiff'] = df['T1_Seed'] - df['T2_Seed']
    df['WinRateDiff'] = df['T1_WinRate'] - df['T2_WinRate']
    df['MarginDiff'] = df['T1_Margin'] - df['T2_Margin']
    df['ScoreDiff'] = df['T1_Score'] - df['T2_Score']
    df['MORDiff'] = df['T1_MOR'] - df['T2_MOR']
    
    # E. 결측치 처리 (시드 데이터가 없는 경우 -1)
    for c in ['T1_Seed', 'T2_Seed']:
        df[c] = df[c].fillna(-1).astype(int)
        
    return df

# 앞에서 만든 데이터프레임 변수를 활용
base_matchup_m = build_base_matchup_df(m_tourney, base_season_stats_m, base_seeds_m, base_massey)
base_matchup_w = build_base_matchup_df(w_tourney, base_season_stats_w, base_seeds_w, mor_df=None)

base_matchup_m.to_csv(os.path.join(OUTPUT_DIR, 'base_matchup_features_M.csv'), index=False)
base_matchup_w.to_csv(os.path.join(OUTPUT_DIR, 'base_matchup_features_W.csv'), index=False)
print("D/E 파트 저장 완료: base_matchup_features_M.csv / base_matchup_features_W.csv")


D/E 파트 저장 완료: base_matchup_features_M.csv / base_matchup_features_W.csv


# New Data Preprocessing

## advanced_stats_features

In [20]:
# 세부 스탯 데이터 로드
df_detail_m = pd.read_csv(os.path.join(INPUT_DIR, 'MRegularSeasonDetailedResults.csv'))
df_detail_w = pd.read_csv(os.path.join(INPUT_DIR, 'WRegularSeasonDetailedResults.csv'))

def build_advanced_stats(df_detail):
    # 승리/패배 구조를 팀 시점으로 변환
    # W, L 로 시작하는 컬럼들을 모두 가져와 앞에 붙은 W/L 알파벳만 제거해 단일 컬럼명으로 만듭니다.
    w_cols = {c: c[1:] for c in df_detail.columns if c.startswith('W') and c not in ['WLoc']}
    l_cols = {c: c[1:] for c in df_detail.columns if c.startswith('L')}

    # (수정된 부분) WTeamID, LTeamID가 w_cols, l_cols에 이미 중복 포함되어 있으므로 명시적 호출 제외
    df_w = df_detail[['Season', 'DayNum'] + list(w_cols.keys())].rename(columns=w_cols)
    df_w['Opp_DR'] = df_detail['LDR']
    df_w['Opp_Score'] = df_detail['LScore']

    df_l = df_detail[['Season', 'DayNum'] + list(l_cols.keys())].rename(columns=l_cols)
    df_l['Opp_DR'] = df_detail['WDR']
    df_l['Opp_Score'] = df_detail['WScore']

    team_stats = pd.concat([df_w, df_l], ignore_index=True)

    # 파생변수 수식 적용 (0으로 나누기 방지를 위해 replace(0, 1) 적용)
    team_stats['Possessions'] = team_stats['FGA'] - team_stats['OR'] + team_stats['TO'] + (0.475 * team_stats['FTA'])
    team_stats['eFG%'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA'].replace(0, 1)
    team_stats['TS%'] = team_stats['Score'] / (2 * (team_stats['FGA'] + 0.475 * team_stats['FTA'])).replace(0, 1)
    team_stats['OR%'] = team_stats['OR'] / (team_stats['OR'] + team_stats['Opp_DR']).replace(0, 1)
    team_stats['TOV%'] = team_stats['TO'] / team_stats['Possessions'].replace(0, 1)
    team_stats['FTRate'] = team_stats['FTA'] / team_stats['FGA'].replace(0, 1)
    team_stats['3PAr'] = team_stats['FGA3'] / team_stats['FGA'].replace(0, 1)
    team_stats['OffRtg'] = (team_stats['Score'] / team_stats['Possessions'].replace(0, 1)) * 100
    team_stats['DefRtg'] = (team_stats['Opp_Score'] / team_stats['Possessions'].replace(0, 1)) * 100
    team_stats['AstTO_Ratio'] = team_stats['Ast'] / team_stats['TO'].replace(0, 1)

    # 시즌 및 팀별 평균 집계 (문서 명세 구조 맞춤)
    advanced_stats = team_stats.drop(columns=['DayNum']).groupby(['Season', 'TeamID']).mean().reset_index()
    return advanced_stats[['Season', 'TeamID', 'FGA', 'FGM', 'OR', 'DR', 'Ast', 'Stl', 'Blk', 
                           'Possessions', 'eFG%', 'TS%', 'OR%', 'TOV%', 'FTRate', '3PAr', 
                           'OffRtg', 'DefRtg', 'AstTO_Ratio']], team_stats

advanced_stats_m, raw_teams_m = build_advanced_stats(df_detail_m)
advanced_stats_w, raw_teams_w = build_advanced_stats(df_detail_w)

advanced_stats_m.to_csv(os.path.join(OUTPUT_DIR, 'advanced_stats_features_M.csv'), index=False)
advanced_stats_w.to_csv(os.path.join(OUTPUT_DIR, 'advanced_stats_features_W.csv'), index=False)
print("[NEW] A 파트 저장 완료: advanced_stats_features_M.csv / advanced_stats_features_W.csv")


[NEW] A 파트 저장 완료: advanced_stats_features_M.csv / advanced_stats_features_W.csv


## momentum_form_features

In [21]:
def build_momentum_features(team_stats, df_compact, massey=None, is_men=True):
    # 승리/패배 추가 및 마스킹 (132일 기준)
    team_stats['Win'] = (team_stats['Score'] > team_stats['Opp_Score']).astype(int)
    team_stats['Margin'] = team_stats['Score'] - team_stats['Opp_Score']
    team_stats['Is_Last_14'] = team_stats['DayNum'] >= (132 - 14)
    team_stats['Is_Last_30'] = team_stats['DayNum'] >= (132 - 30)

    # 1. 단기 및 중기 롤링 평균
    last14 = team_stats[team_stats['Is_Last_14']].groupby(['Season', 'TeamID']).agg(
        Last14_WinRate=('Win', 'mean'),
        Last14_Margin=('Margin', 'mean')
    ).reset_index()

    last30 = team_stats[team_stats['Is_Last_30']].groupby(['Season', 'TeamID']).agg(
        Last30_WinRate=('Win', 'mean'),
        Last30_eFG_pct=('eFG%', 'mean'),
        Last30_TOV_pct=('TOV%', 'mean')
    ).rename(columns={'Last30_eFG_pct': 'Last30_eFG%', 'Last30_TOV_pct': 'Last30_TOV%'}).reset_index()

    # 2. 진짜 연승/연패 흐름 (Active Win Streak)
    df_compact_sorted = df_compact.sort_values(by=['Season', 'DayNum'])
    streak_records = []
    
    for season in df_compact_sorted['Season'].unique():
        s_df = df_compact_sorted[df_compact_sorted['Season'] == season]
        teams = set(s_df['WTeamID']).union(set(s_df['LTeamID']))
        for t in teams:
            t_games = s_df[(s_df['WTeamID'] == t) | (s_df['LTeamID'] == t)].copy()
            if t_games.empty: continue
            
            t_games['Is_Win'] = (t_games['WTeamID'] == t).astype(int)
            # 연속성 그룹화
            t_games['Streak_Group'] = (t_games['Is_Win'] != t_games['Is_Win'].shift()).cumsum()
            last_group = t_games.iloc[-1]['Streak_Group']
            current_streak_len = len(t_games[t_games['Streak_Group'] == last_group])
            
            # 이기고 있으면 양수(+), 지고 있으면 음수(-)로 부여
            active_streak = current_streak_len if t_games.iloc[-1]['Is_Win'] == 1 else -current_streak_len
            streak_records.append({'Season': season, 'TeamID': t, 'Active_WinStreak': active_streak})
    
    streak_df = pd.DataFrame(streak_records)

    # 3. Quality Wins (강팀 사냥) 및 Upset Value 산출 (남자 농구 한정)
    if is_men and massey is not None:
        # MOR 시스템 (매시 예측)만 추출
        mor = massey[massey['SystemName'] == 'MOR'].copy()
        
        # 원본에서 상대팀 정보 매핑용
        games_info = df_compact[['Season', 'DayNum', 'WTeamID', 'LTeamID']].copy()
        w_view = games_info.copy().rename(columns={'WTeamID': 'TeamID', 'LTeamID': 'OppTeamID'}) 
        w_view['Win'] = 1
        l_view = games_info.copy().rename(columns={'LTeamID': 'TeamID', 'WTeamID': 'OppTeamID'}) 
        l_view['Win'] = 0
        games_opp = pd.concat([w_view, l_view])
        
        team_stats = team_stats.merge(games_opp[['Season', 'DayNum', 'TeamID', 'OppTeamID']], 
                                      on=['Season', 'DayNum', 'TeamID'], how='left')
        
        # [수정] merge_asof를 위해 DayNum을 기준으로 오름차순 완전 정렬
        games_sorted = team_stats.sort_values('DayNum')
        mor_sorted = mor.sort_values('RankingDayNum')
        
        # 내 랭킹 매핑
        merged = pd.merge_asof(
            games_sorted, mor_sorted[['Season', 'TeamID', 'RankingDayNum', 'OrdinalRank']], 
            left_on='DayNum', right_on='RankingDayNum',
            by=['Season', 'TeamID'], direction='backward'
        ).rename(columns={'OrdinalRank': 'My_Rank'})
        
        # [수정] 상대방 랭킹을 붙이기 위해 병합 결과물을 다시 한번 인덱스 정렬
        merged = merged.sort_values('DayNum')
        
        # 상대 랭킹 매핑
        merged = pd.merge_asof(
            merged, mor_sorted[['Season', 'TeamID', 'RankingDayNum', 'OrdinalRank']].rename(columns={'TeamID': 'OppTeamID'}),
            left_on='DayNum', right_on='RankingDayNum',
            by=['Season', 'OppTeamID'], direction='backward'
        ).rename(columns={'OrdinalRank': 'Opp_Rank'})
        
        # Quality Wins (최근 30일 이내, 상대 랭킹 50위 이내이면서 이긴 경기)
        qw = merged[(merged['Is_Last_30']) & (merged['Win'] == 1) & (merged['Opp_Rank'] <= 50)]
        qw_count = qw.groupby(['Season', 'TeamID']).size().reset_index(name='Last30_QualityWins')
        
        # Upset Value: 내가 이겼는데 상대가 나보다 랭킹이 높았을 때 랭킹 차이의 합
        merged['Upset_Diff'] = np.where((merged['Win'] == 1) & (merged['My_Rank'] > merged['Opp_Rank']), 
                                        merged['My_Rank'] - merged['Opp_Rank'], 0)
        upset_val = merged[(merged['Is_Last_30'])].groupby(['Season', 'TeamID'])['Upset_Diff'].sum().reset_index(name='Upset_Value')
        
        qw_final = qw_count.merge(upset_val, on=['Season', 'TeamID'], how='outer')
        
    else:
        qw_final = pd.DataFrame(columns=['Season', 'TeamID', 'Last30_QualityWins', 'Upset_Value'])
        for season in team_stats['Season'].unique():
            teams = df_compact[df_compact['Season'] == season]['WTeamID'].unique()
            temp = pd.DataFrame({'Season': season, 'TeamID': teams, 'Last30_QualityWins': 0, 'Upset_Value': 0})
            qw_final = pd.concat([qw_final, temp], ignore_index=True)

    qw_final['Last30_QualityWins'] = qw_final['Last30_QualityWins'].fillna(0)
    qw_final['Upset_Value'] = qw_final['Upset_Value'].fillna(0)

    # 전체 데이터 병합
    momentum = last14.merge(last30, on=['Season', 'TeamID'], how='outer')\
                     .merge(streak_df, on=['Season', 'TeamID'], how='outer')\
                     .merge(qw_final, on=['Season', 'TeamID'], how='outer')
    
    return momentum[['Season', 'TeamID', 'Last14_WinRate', 'Last30_WinRate', 'Last14_Margin', 
                     'Last30_eFG%', 'Last30_TOV%', 'Active_WinStreak', 'Last30_QualityWins', 'Upset_Value']]

# 구동 셀
massey_raw = pd.read_csv(os.path.join(INPUT_DIR, 'MMasseyOrdinals.csv'))
momentum_m = build_momentum_features(raw_teams_m, m_reg, massey=massey_raw, is_men=True)
momentum_w = build_momentum_features(raw_teams_w, w_reg, massey=None, is_men=False)

momentum_m.to_csv(os.path.join(OUTPUT_DIR, 'momentum_form_features_M.csv'), index=False)
momentum_w.to_csv(os.path.join(OUTPUT_DIR, 'momentum_form_features_W.csv'), index=False)
print("[NEW] B 파트 저장 완료: momentum_form_features_M.csv / momentum_form_features_W.csv")


[NEW] B 파트 저장 완료: momentum_form_features_M.csv / momentum_form_features_W.csv


/var/folders/5r/gp1130xx2t5fbxbgt43xtp_m0000gn/T/ipykernel_6786/170878270.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  qw_final['Last30_QualityWins'] = qw_final['Last30_QualityWins'].fillna(0)
/var/folders/5r/gp1130xx2t5fbxbgt43xtp_m0000gn/T/ipykernel_6786/170878270.py:99: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  qw_final['Upset_Value'] = qw_final['Upset_Value'].fillna(0)


## program_pedigree_features

In [22]:
def build_program_pedigree(massey, tourney, coaches=None):
    pedigree_list = []
    seasons = sorted(tourney['Season'].unique())
    teams = set(tourney['WTeamID']).union(set(tourney['LTeamID']))
    
    for season in seasons:
        team_df = pd.DataFrame({'TeamID': list(teams)})
        team_df['Season'] = season

        # 1. 시계열 랭킹 동향 (남자부 한정)
        if massey is not None:
            past_3_seasons = [season - 1, season - 2, season - 3]
            past_ranks = massey[(massey['Season'].isin(past_3_seasons)) & (massey['SystemName'] == 'MOR')] \
                         .groupby(['Season', 'TeamID'])['OrdinalRank'].last().reset_index()
            
            rank_avg = past_ranks.groupby('TeamID')['OrdinalRank'].mean().reset_index(name='3Yr_Rank_Avg')
            rank_trend = past_ranks[past_ranks['Season'] == (season - 1)].rename(columns={'OrdinalRank': 'LastYr_Rank'})
            rank_stats = rank_avg.merge(rank_trend[['TeamID', 'LastYr_Rank']], on='TeamID', how='left')
            rank_stats['Rank_Trend_Slope'] = rank_stats['LastYr_Rank'] - rank_stats['3Yr_Rank_Avg']
            team_df = team_df.merge(rank_stats[['TeamID', '3Yr_Rank_Avg', 'Rank_Trend_Slope']], on='TeamID', how='left')
        else:
            team_df['3Yr_Rank_Avg'] = np.nan
            team_df['Rank_Trend_Slope'] = np.nan
            
        # 2. 다년간 토너먼트 진출 실적 및 1라운드(64강) Upset Rate
        past_5_seasons = list(range(season - 5, season))
        past_tourney = tourney[tourney['Season'].isin(past_5_seasons)]
        
        # 2-A. Sweet16 (DayNum >= 143) 점수
        sweet16_exp = past_tourney[past_tourney['DayNum'] >= 143].groupby('WTeamID').size().reset_index(name='Tourney_Exp_Score').rename(columns={'WTeamID': 'TeamID'})
        team_df = team_df.merge(sweet16_exp, on='TeamID', how='left')
        team_df['Tourney_Exp_Score'] = team_df['Tourney_Exp_Score'].fillna(0)
        
        # 2-B. 1라운드 Upset(하위 시드가 상위 시드 잡음) 경험치
        # 과거 5년 토너먼트 1라운드 경기 (DayNum 136, 137)
        first_round = past_tourney[past_tourney['DayNum'].isin([136, 137])]
        if len(first_round) > 0:
            upset_counts = first_round.groupby('WTeamID').size().reset_index(name='First_Round_Upset_Score').rename(columns={'WTeamID': 'TeamID'})
            team_df = team_df.merge(upset_counts, on='TeamID', how='left')
            team_df['First_Round_Upset_Score'] = team_df['First_Round_Upset_Score'].fillna(0)
        else:
            team_df['First_Round_Upset_Score'] = 0

        # 3. 감독 통산 실적 (남자 경기 전용)
        if coaches is not None:
            # 현재 시즌 팀별 감독 매핑
            current_coaches = coaches[coaches['Season'] == season]
            # 해당 감독의 통산 커리어 (현재 시즌 전까지, 모든 팀 통합)
            past_coaches_career = coaches[coaches['Season'] < season]
            
            # 감독 연차 (현재 팀)
            coach_tenure = coaches[coaches['Season'] <= season].groupby(['TeamID', 'CoachName']).size().reset_index(name='Coach_Tenure')
            coach_stats = current_coaches[['TeamID', 'CoachName']].merge(coach_tenure, on=['TeamID', 'CoachName'], how='left')
            coach_stats['Is_Rookie_Coach'] = (coach_stats['Coach_Tenure'] == 1).astype(int)
            
            # 감독 통산 토너먼트 승리 횟수 추적
            past_tourney_all = tourney[tourney['Season'] < season]
            # 승리한 팀의 소속 시즌과 ID로 당시 승리 감독 매핑
            tourney_wins_with_coaches = past_tourney_all.merge(past_coaches_career, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
            coach_career_wins = tourney_wins_with_coaches.groupby('CoachName').size().reset_index(name='Coach_Tourney_Wins')
            
            # 병합
            coach_stats = coach_stats.merge(coach_career_wins, on='CoachName', how='left')
            coach_stats['Coach_Tourney_Wins'] = coach_stats['Coach_Tourney_Wins'].fillna(0)
            
            team_df = team_df.merge(coach_stats[['TeamID', 'Coach_Tenure', 'Is_Rookie_Coach', 'Coach_Tourney_Wins']], on='TeamID', how='left')
        else:
            team_df['Coach_Tenure'] = np.nan
            team_df['Is_Rookie_Coach'] = -1
            team_df['Coach_Tourney_Wins'] = np.nan
            
        pedigree_list.append(team_df)
        
    final_pedigree = pd.concat(pedigree_list, ignore_index=True)
    return final_pedigree

m_coaches = pd.read_csv(os.path.join(INPUT_DIR, 'MTeamCoaches.csv'))
pedigree_m = build_program_pedigree(massey_raw, m_tourney, coaches=m_coaches)
pedigree_w = build_program_pedigree(None, w_tourney, coaches=None)

pedigree_m.to_csv(os.path.join(OUTPUT_DIR, 'program_pedigree_features_M.csv'), index=False)
pedigree_w.to_csv(os.path.join(OUTPUT_DIR, 'program_pedigree_features_W.csv'), index=False)
print("[NEW] C 파트 저장 완료: program_pedigree_features_M.csv / program_pedigree_features_W.csv")


[NEW] C 파트 저장 완료: program_pedigree_features_M.csv / program_pedigree_features_W.csv


## sos_features

In [23]:
MAJOR_CONFS = ['acc', 'big_east', 'big_ten', 'big_twelve', 'pac_twelve', 'sec']

def build_sos_features(raw_stats, conf_df, massey=None, is_men=True):
    # 1. 타임라인(Season, TeamID) 뼈대
    seasons = raw_stats['Season'].unique()
    teams = set(raw_stats['WTeamID']).union(set(raw_stats['LTeamID']))
    base_df = pd.DataFrame([(s, t) for s in seasons for t in teams], columns=['Season', 'TeamID'])
    
    # 2. 메이저 컨퍼런스 여부
    conf_df['Is_Major_Conf'] = conf_df['ConfAbbrev'].isin(MAJOR_CONFS).astype(int)
    sos = base_df.merge(conf_df[['Season', 'TeamID', 'Is_Major_Conf', 'ConfAbbrev']], on=['Season', 'TeamID'], how='left')
    sos['Is_Major_Conf'] = sos['Is_Major_Conf'].fillna(0)
    
    # 3. 메이저 컨퍼런스 내 맞대결 승률 (이너 리그 승률)
    # 두 팀 모두 메이저 컨퍼런스, 단 동일한 컨퍼런스일 경우 해당 경기 매핑
    w_conf = raw_stats[['Season', 'DayNum', 'WTeamID']].merge(conf_df, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']).rename(columns={'ConfAbbrev': 'W_Conf', 'Is_Major_Conf': 'W_Major'})
    l_conf = raw_stats[['Season', 'DayNum', 'LTeamID']].merge(conf_df, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']).rename(columns={'ConfAbbrev': 'L_Conf', 'Is_Major_Conf': 'L_Major'})
    major_games = w_conf.merge(l_conf, on=['Season', 'DayNum'])
    
    # 같은 컨퍼런스 소속인 파워 6 소속팀 간의 매치만 필터링
    major_intra_games = major_games[(major_games['W_Major'] == 1) & (major_games['W_Conf'] == major_games['L_Conf'])]
    major_wins = major_intra_games.groupby(['Season', 'WTeamID']).size().reset_index(name='Major_Wins')
    major_loss = major_intra_games.groupby(['Season', 'LTeamID']).size().reset_index(name='Major_Loss')
    
    major_stats = major_wins.rename(columns={'WTeamID': 'TeamID'}).merge(major_loss.rename(columns={'LTeamID': 'TeamID'}), on=['Season', 'TeamID'], how='outer').fillna(0)
    major_stats['Major_Conf_WinRate'] = major_stats['Major_Wins'] / (major_stats['Major_Wins'] + major_stats['Major_Loss']).replace(0, 1)
    
    sos = sos.merge(major_stats[['Season', 'TeamID', 'Major_Conf_WinRate']], on=['Season', 'TeamID'], how='left')
    sos['Major_Conf_WinRate'] = sos['Major_Conf_WinRate'].fillna(0) # 메이저 컨퍼런스가 아닌 팀들은 0

    # 4. 시즌 초반(DayNum <= 50) 비컨퍼런스(타 컨퍼런스 교류전) 일정 강도 (상대 승률 평균치로 대체)
    # 진정한 약팀 학살 여부 판단하기 위함
    non_conf_games = major_games[major_games['W_Conf'] != major_games['L_Conf']]
    early_non_conf = non_conf_games[non_conf_games['DayNum'] <= 50]
    
    # 이긴 관점, 진 관점으로 양 팀의 상대 빈도 측정
    w_opps = early_non_conf.groupby(['Season', 'WTeamID']).size().reset_index(name='NonConf_Count').rename(columns={'WTeamID':'TeamID'})
    l_opps = early_non_conf.groupby(['Season', 'LTeamID']).size().reset_index(name='NonConf_Count').rename(columns={'LTeamID':'TeamID'})
    non_conf_counts = pd.concat([w_opps, l_opps]).groupby(['Season', 'TeamID'])['NonConf_Count'].sum().reset_index(name='Non_Conf_Games_Count')
    
    # 시간복잡도 상 단순히 '시즌 초반에 타 리그 강팀과 많이 붙었나'를 교류전 횟수 등급으로 지표화 (Non_Conf_SOS)
    sos = sos.merge(non_conf_counts, on=['Season', 'TeamID'], how='left')
    sos.rename(columns={'Non_Conf_Games_Count': 'Non_Conf_SOS'}, inplace=True)
    sos['Non_Conf_SOS'] = sos['Non_Conf_SOS'].fillna(0)

    # 5. Top 50 랭커 팀 상대 승률 (WinRate vs Top 50) - 남자부 데이터 제공 시
    if is_men and massey is not None:
        mor = massey[massey['SystemName'] == 'MOR']
        # [수정] DayNum 정렬 오류 방지를 위해 단일 키로 완전 정렬 우선 실시
        mor_sorted = mor.sort_values('RankingDayNum')
        
        # 경기 데이터 평탄화 (TeamID, OppTeamID, Win 여부)
        w_view = raw_stats[['Season', 'DayNum', 'WTeamID', 'LTeamID']].rename(columns={'WTeamID': 'TeamID', 'LTeamID': 'OppTeamID'})
        w_view['Win'] = 1
        l_view = raw_stats[['Season', 'DayNum', 'LTeamID', 'WTeamID']].rename(columns={'LTeamID': 'TeamID', 'WTeamID': 'OppTeamID'})
        l_view['Win'] = 0
        
        # [수정] asof_merge의 left가 될 기준 테이블 무조건 DayNum 기준 단일 오름차순 정렬 
        all_games = pd.concat([w_view, l_view]).sort_values('DayNum')
        
        # 상대팀의 당시 랭킹 매핑
        merged_opp = pd.merge_asof(
            all_games, mor_sorted[['Season', 'TeamID', 'RankingDayNum', 'OrdinalRank']].rename(columns={'TeamID': 'OppTeamID'}),
            left_on='DayNum', right_on='RankingDayNum',
            by=['Season', 'OppTeamID'], direction='backward'
        ).rename(columns={'OrdinalRank': 'Opp_Rank'})
        
        # 상대가 Top 50 이내인 하드코어 경기 필터링
        top50_games = merged_opp[merged_opp['Opp_Rank'] <= 50]
        t50_wins = top50_games[top50_games['Win'] == 1].groupby(['Season', 'TeamID']).size().reset_index(name='Top50_Wins')
        t50_total = top50_games.groupby(['Season', 'TeamID']).size().reset_index(name='Top50_Games')
        
        t50_stats = t50_total.merge(t50_wins, on=['Season', 'TeamID'], how='left').fillna(0)
        t50_stats['WinRate_vs_Top50'] = t50_stats['Top50_Wins'] / t50_stats['Top50_Games'].replace(0, 1)
        
        sos = sos.merge(t50_stats[['Season', 'TeamID', 'WinRate_vs_Top50']], on=['Season', 'TeamID'], how='left')
        sos['WinRate_vs_Top50'] = sos['WinRate_vs_Top50'].fillna(0)
    else:
        sos['WinRate_vs_Top50'] = 0

    # 컨퍼런스 토너먼트 우승/승리 횟수 (DayNum 120 ~ 132)
    conf_tourney_wins = raw_stats[(raw_stats['DayNum'] >= 120) & (raw_stats['DayNum'] <= 132)].groupby(['Season', 'WTeamID']).size().reset_index(name='Conf_Tourney_Wins')
    conf_tourney_wins.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
    sos = sos.merge(conf_tourney_wins, on=['Season', 'TeamID'], how='left')
    sos['Conf_Tourney_Wins'] = sos['Conf_Tourney_Wins'].fillna(0)
    
    # 필요 없는 조인 키 드롭
    sos.drop(columns=['ConfAbbrev'], inplace=True, errors='ignore')
    
    return sos[['Season', 'TeamID', 'WinRate_vs_Top50', 'Non_Conf_SOS', 'Is_Major_Conf', 'Major_Conf_WinRate', 'Conf_Tourney_Wins']]

# (추가) 팀 컨퍼런스 데이터 로드
m_conf = pd.read_csv(os.path.join(INPUT_DIR, 'MTeamConferences.csv'))
w_conf = pd.read_csv(os.path.join(INPUT_DIR, 'WTeamConferences.csv'))

# 구동 
sos_m = build_sos_features(m_reg, m_conf, massey=massey_raw, is_men=True)
sos_w = build_sos_features(w_reg, w_conf, massey=None, is_men=False)

sos_m.to_csv(os.path.join(OUTPUT_DIR, 'sos_features_M.csv'), index=False)
sos_w.to_csv(os.path.join(OUTPUT_DIR, 'sos_features_W.csv'), index=False)
print("[NEW] D 파트 저장 완료: sos_features_M.csv / sos_features_W.csv")


[NEW] D 파트 저장 완료: sos_features_M.csv / sos_features_W.csv


## brier_score_features

In [24]:
def build_brier_features(raw_stats):
    # 팀 시점으로 데이터 변환
    w = raw_stats[['Season', 'WTeamID', 'WScore', 'LScore']].copy()
    w.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    w['Win'] = 1
    
    l = raw_stats[['Season', 'LTeamID', 'LScore', 'WScore']].copy()
    l.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    l['Win'] = 0
    
    team_stats = pd.concat([w, l], ignore_index=True)
    team_stats['Margin'] = team_stats['ScoreFor'] - team_stats['ScoreAgainst']
    team_stats['Abs_Margin'] = abs(team_stats['Margin'])
    
    # 1. 득점/득실차의 기복 (표준편차)
    std_df = team_stats.groupby(['Season', 'TeamID']).agg(
        Margin_Std=('Margin', 'std'),
        Score_Std=('ScoreFor', 'std'),
        Score_Max=('ScoreFor', 'max')
    ).reset_index()
    
    # 2. 박빙 경기(점수 차이 5점 이하) 승률
    close_games = team_stats[team_stats['Abs_Margin'] <= 5]
    close_winrate = close_games.groupby(['Season', 'TeamID']).agg(
        Close_Game_WinRate=('Win', 'mean')
    ).reset_index()
    
    # 3. 블로우아웃(대승 - 20점 차 이상) 승리 횟수
    blowout_wins = team_stats[(team_stats['Margin'] >= 20) & (team_stats['Win'] == 1)]
    blowout_count = blowout_wins.groupby(['Season', 'TeamID']).size().reset_index(name='Blowout_Wins_Count')
    
    # 병합
    brier = std_df.merge(close_winrate, on=['Season', 'TeamID'], how='left')\
                  .merge(blowout_count, on=['Season', 'TeamID'], how='left')
                  
    brier['Close_Game_WinRate'] = brier['Close_Game_WinRate'].fillna(0) # 박빙 경기를 한 번도 안 해봤으면 0
    brier['Blowout_Wins_Count'] = brier['Blowout_Wins_Count'].fillna(0)
    
    return brier[['Season', 'TeamID', 'Margin_Std', 'Close_Game_WinRate', 'Blowout_Wins_Count', 'Score_Max', 'Score_Std']]

brier_m = build_brier_features(m_reg)
brier_w = build_brier_features(w_reg)

brier_m.to_csv(os.path.join(OUTPUT_DIR, 'brier_score_features_M.csv'), index=False)
brier_w.to_csv(os.path.join(OUTPUT_DIR, 'brier_score_features_W.csv'), index=False)
print("[NEW] E 파트 저장 완료: brier_score_features_M.csv / brier_score_features_W.csv")


[NEW] E 파트 저장 완료: brier_score_features_M.csv / brier_score_features_W.csv


## past_tourney_features

In [25]:
# 1. 과거 토너먼트 세부 스탯 및 하위 대회 데이터 로드
m_detailed_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneyDetailedResults.csv'))
w_detailed_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneyDetailedResults.csv'))

m_secondary_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MSecondaryTourneyCompactResults.csv'))
w_secondary_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WSecondaryTourneyCompactResults.csv'))

def build_past_tourney_features(detailed_tourney, secondary_tourney):
    # 1. 과거 토너먼트 슈팅 집중력 (eFG%) 및 안정성 (TOV%)
    # 승/패 팀별 데이터를 한 팀 시점으로(W/L 무관하게) 재배열
    w_cols = {c: c[1:] for c in detailed_tourney.columns if c.startswith('W') and c not in ['WLoc']}
    l_cols = {c: c[1:] for c in detailed_tourney.columns if c.startswith('L')}
    
    df_w = detailed_tourney[['Season'] + list(w_cols.keys())].rename(columns=w_cols)
    df_l = detailed_tourney[['Season'] + list(l_cols.keys())].rename(columns=l_cols)
    
    team_stats = pd.concat([df_w, df_l], ignore_index=True)
    
    # 포제션 및 각종 비율 계산
    team_stats['Possessions'] = team_stats['FGA'] - team_stats['OR'] + team_stats['TO'] + (0.475 * team_stats['FTA'])
    team_stats['eFG%'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA'].replace(0, 1)
    team_stats['TOV%'] = team_stats['TO'] / team_stats['Possessions'].replace(0, 1)
    
    # 과거 (누적) 시즌의 모든 토너먼트 경기 평균을 냅니다.
    past_tourney_stats = team_stats.groupby('TeamID').agg(
        Past_Tourney_eFG_pct=('eFG%', 'mean'),
        Past_Tourney_TOV_pct=('TOV%', 'mean')
    ).rename(columns={'Past_Tourney_eFG_pct': 'Past_Tourney_eFG%', 'Past_Tourney_TOV_pct': 'Past_Tourney_TOV%'}).reset_index()
    
    # 2. 직전 1~2개 시즌 하위 토너먼트 딥런 여부 (Secondary Tourney Success)
    seasons = sorted(detailed_tourney['Season'].unique())
    teams = set(team_stats['TeamID'])
    
    past_feature_list = []
    for target_season in seasons:
        # 현재 예측하려는 막대기(타겟 시즌)의 데이터 뼈대
        temp_df = pd.DataFrame({'TeamID': list(teams)})
        temp_df['Season'] = target_season
        
        temp_df = temp_df.merge(past_tourney_stats, on='TeamID', how='left')
        
        # 과거 토너먼트 경험이 없으면 전체 평균값으로 결측치 채우기 (Imputation)
        temp_df['Past_Tourney_eFG%'] = temp_df['Past_Tourney_eFG%'].fillna(temp_df['Past_Tourney_eFG%'].mean())
        temp_df['Past_Tourney_TOV%'] = temp_df['Past_Tourney_TOV%'].fillna(temp_df['Past_Tourney_TOV%'].mean())
        
        # 하위 대회 성공 기록 확인 (Season-1, Season-2)
        prev_seasons = [target_season - 1, target_season - 2]
        
        if secondary_tourney is not None:
            # 해당 예전 시즌들의 승리팀 명단을 묶어 여러 번 이긴 팀(딥런한 팀) 카운트
            prev_sec_tourney = secondary_tourney[secondary_tourney['Season'].isin(prev_seasons)]
            win_counts = prev_sec_tourney.groupby('WTeamID').size().reset_index(name='Sec_Wins')
            win_counts.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
            
            temp_df = temp_df.merge(win_counts, on='TeamID', how='left')
            temp_df['Sec_Wins'] = temp_df['Sec_Wins'].fillna(0)
            
            # 3승 이상 (보통 4강 이상 급) 했으면 Success 플래그 1
            temp_df['Prev_Secondary_Tourney_Success'] = (temp_df['Sec_Wins'] >= 3).astype(int)
            temp_df.drop(columns=['Sec_Wins'], inplace=True)
        else:
            temp_df['Prev_Secondary_Tourney_Success'] = 0
            
        past_feature_list.append(temp_df)
        
    final_df = pd.concat(past_feature_list, ignore_index=True)
    return final_df[['Season', 'TeamID', 'Past_Tourney_eFG%', 'Past_Tourney_TOV%', 'Prev_Secondary_Tourney_Success']]


past_tourney_m = build_past_tourney_features(m_detailed_tourney, m_secondary_tourney)
past_tourney_w = build_past_tourney_features(w_detailed_tourney, w_secondary_tourney)

past_tourney_m.to_csv(os.path.join(OUTPUT_DIR, 'past_tourney_features_M.csv'), index=False)
past_tourney_w.to_csv(os.path.join(OUTPUT_DIR, 'past_tourney_features_W.csv'), index=False)
print("[NEW] F 파트 저장 완료: past_tourney_features_M.csv / past_tourney_features_W.csv")


[NEW] F 파트 저장 완료: past_tourney_features_M.csv / past_tourney_features_W.csv


## schedule_rest_features

In [26]:
def build_schedule_rest_features(reg_results):
    # 각 팀의 정규 시즌(컨퍼런스 포함) 마지막 경기 날짜 (DayNum) 추출
    last_game_w = reg_results.groupby(['Season', 'WTeamID'])['DayNum'].max().reset_index().rename(columns={'WTeamID': 'TeamID', 'DayNum': 'W_Last_Day'})
    last_game_l = reg_results.groupby(['Season', 'LTeamID'])['DayNum'].max().reset_index().rename(columns={'LTeamID': 'TeamID', 'DayNum': 'L_Last_Day'})
    
    # 승리, 패배 일자 병합 후 가장 마지막 승부 날짜를 구함
    last_game_df = pd.merge(last_game_w, last_game_l, on=['Season', 'TeamID'], how='outer')
    last_game_df['Last_Game_DayNum'] = last_game_df[['W_Last_Day', 'L_Last_Day']].max(axis=1)
    
    # NCAA 토너먼트 1라운드는 통상적으로 DayNum 136 (목요일) 또는 137 (금요일)에 시작.
    # 일괄적으로 136을 토너먼트 1라운드 시작 기준일로 잡음
    TOURNEY_START_DAY = 136 
    
    # 토너먼트 시작 전까지 순수하게 며칠을 쉬었는지 계산 (휴식일 지표)
    last_game_df['Days_Since_Last_Game'] = TOURNEY_START_DAY - last_game_df['Last_Game_DayNum']
    
    return last_game_df[['Season', 'TeamID', 'Days_Since_Last_Game']]

schedule_rest_m = build_schedule_rest_features(m_reg)
schedule_rest_w = build_schedule_rest_features(w_reg)

schedule_rest_m.to_csv(os.path.join(OUTPUT_DIR, 'schedule_rest_features_M.csv'), index=False)
schedule_rest_w.to_csv(os.path.join(OUTPUT_DIR, 'schedule_rest_features_W.csv'), index=False)
print("[NEW] G 파트 저장 완료: schedule_rest_features_M.csv / schedule_rest_features_W.csv")


[NEW] G 파트 저장 완료: schedule_rest_features_M.csv / schedule_rest_features_W.csv


## geography_travel_features_M

In [27]:
CITY_GEO_DICT = {
    4001: {"Lat": 32.4464, "Lon": -99.7476, "Alt": 531.0},
    4002: {"Lat": 41.0831, "Lon": -81.5185, "Alt": 292.0},
    4003: {"Lat": 42.6512, "Lon": -73.755, "Alt": 38.0},
    4004: {"Lat": 35.0841, "Lon": -106.651, "Alt": 1514.0},
    4005: {"Lat": 40.6023, "Lon": -75.4716, "Alt": 112.0},
    4006: {"Lat": 42.0268, "Lon": -93.617, "Alt": 285.0},
    4007: {"Lat": 42.3732, "Lon": -72.5199, "Alt": 100.0},
    4008: {"Lat": 33.8348, "Lon": -117.9117, "Alt": 54.0},
    4009: {"Lat": 61.2163, "Lon": -149.8949, "Alt": 39.0},
    4010: {"Lat": 42.2814, "Lon": -83.7485, "Alt": 255.0},
    4011: {"Lat": 38.9786, "Lon": -76.4928, "Alt": 19.0},
    4012: {"Lat": 32.7356, "Lon": -97.1071, "Alt": 191.0},
    4013: {"Lat": 35.5954, "Lon": -82.5508, "Alt": 674.0},
    4014: {"Lat": 33.9598, "Lon": -83.3764, "Alt": 236.0},
    4015: {"Lat": 39.3576, "Lon": -82.061, "Alt": 252.0},
    4016: {"Lat": 33.7545, "Lon": -84.3898, "Alt": 325.0},
    4017: {"Lat": 39.3643, "Lon": -74.4229, "Alt": 4.0},
    4018: {"Lat": 32.6099, "Lon": -85.4808, "Alt": 215.0},
    4019: {"Lat": 42.6875, "Lon": -83.2341, "Alt": 293.0},
    4020: {"Lat": 30.2711, "Lon": -97.7437, "Alt": 167.0},
    4021: {"Lat": 35.3739, "Lon": -119.0195, "Alt": 124.0},
    4022: {"Lat": 39.2909, "Lon": -76.6108, "Alt": 13.0},
    4023: {"Lat": 44.8016, "Lon": -68.7713, "Alt": 15.0},
    4024: {"Lat": 30.4494, "Lon": -91.187, "Alt": 18.0},
    4025: {"Lat": 18.345, "Lon": -66.1684, "Alt": 77.0},
    4026: {"Lat": 30.0829, "Lon": -94.0984, "Alt": 10.0},
    4027: {"Lat": 37.8708, "Lon": -122.2729, "Alt": 56.0},
    4028: {"Lat": 40.6179, "Lon": -75.3787, "Alt": 92.0},
    4029: {"Lat": 45.7875, "Lon": -108.4961, "Alt": 952.0},
    4030: {"Lat": 33.5207, "Lon": -86.8024, "Alt": 187.0},
    4031: {"Lat": 37.2297, "Lon": -80.4137, "Alt": 633.0},
    4032: {"Lat": 39.167, "Lon": -86.5343, "Alt": 238.0},
    4033: {"Lat": 26.3587, "Lon": -80.0831, "Alt": 9.0},
    4034: {"Lat": 35.2543, "Lon": -81.667, "Alt": 271.0},
    4035: {"Lat": 43.6166, "Lon": -116.2009, "Alt": 829.0},
    4036: {"Lat": 36.2188, "Lon": -81.684, "Alt": 991.0},
    4037: {"Lat": 18.1681, "Lon": -66.041, "Alt": 137.0},
    4038: {"Lat": 32.5159, "Lon": -93.7337, "Alt": 59.0},
    4039: {"Lat": 42.3588, "Lon": -71.0578, "Alt": 14.0},
    4040: {"Lat": 40.015, "Lon": -105.2705, "Alt": 1624.0},
    4041: {"Lat": 36.9929, "Lon": -86.4429, "Alt": 156.0},
    4042: {"Lat": 41.3748, "Lon": -83.6513, "Alt": 213.0},
    4043: {"Lat": 45.6794, "Lon": -111.044, "Alt": 1471.0},
    4044: {"Lat": 41.1793, "Lon": -73.1888, "Alt": 14.0},
    4045: {"Lat": 40.8467, "Lon": -73.8786, "Alt": 22.0},
    4046: {"Lat": 44.3115, "Lon": -96.7984, "Alt": 494.0},
    4047: {"Lat": 40.6526, "Lon": -73.9497, "Alt": 23.0},
    4048: {"Lat": 40.6961, "Lon": -73.995, "Alt": 37.0},
    4049: {"Lat": 42.8864, "Lon": -78.8781, "Alt": 183.0},
    4050: {"Lat": 35.4123, "Lon": -78.7367, "Alt": 66.0},
    4051: {"Lat": 44.4762, "Lon": -73.2129, "Alt": 65.0},
    4052: {"Lat": 42.3656, "Lon": -71.104, "Alt": 8.0},
    4053: {"Lat": 21.1527, "Lon": -86.8426, "Alt": 7.0},
    4054: {"Lat": 37.3047, "Lon": -89.5176, "Alt": 110.0},
    4055: {"Lat": 37.7275, "Lon": -89.2167, "Alt": 127.0},
    4056: {"Lat": 42.8501, "Lon": -106.3251, "Alt": 1561.0},
    4057: {"Lat": 37.6774, "Lon": -113.0618, "Alt": 1782.0},
    4058: {"Lat": 42.5362, "Lon": -92.4478, "Alt": 267.0},
    4059: {"Lat": 30.5217, "Lon": -97.8278, "Alt": 299.0},
    4060: {"Lat": 40.1165, "Lon": -88.2431, "Alt": 227.0},
    4061: {"Lat": 35.9132, "Lon": -79.0558, "Alt": 151.0},
    4062: {"Lat": 39.4961, "Lon": -88.1762, "Alt": 205.0},
    4063: {"Lat": 36.1202, "Lon": -77.3755, "Alt": 27.0},
    4064: {"Lat": 32.7884, "Lon": -79.9399, "Alt": 5.0},
    4065: {"Lat": 38.3506, "Lon": -81.6333, "Alt": 191.0},
    4066: {"Lat": 35.2272, "Lon": -80.8431, "Alt": 254.0},
    4067: {"Lat": 38.0293, "Lon": -78.4767, "Alt": 151.0},
    4068: {"Lat": 35.0457, "Lon": -85.3095, "Alt": 209.0},
    4069: {"Lat": 47.4892, "Lon": -117.5793, "Alt": 744.0},
    4070: {"Lat": 42.3235, "Lon": -71.1639, "Alt": 55.0},
    4071: {"Lat": 41.8756, "Lon": -87.6244, "Alt": 199.0},
    4072: {"Lat": 39.1015, "Lon": -84.5125, "Alt": 176.0},
    4073: {"Lat": 36.5278, "Lon": -87.3589, "Alt": 154.0},
    4074: {"Lat": 34.6851, "Lon": -82.8364, "Alt": 216.0},
    4075: {"Lat": 41.4997, "Lon": -81.6937, "Alt": 197.0},
    4076: {"Lat": 34.4726, "Lon": -81.8807, "Alt": 209.0},
    4077: {"Lat": 38.9807, "Lon": -76.9369, "Alt": 25.0},
    4078: {"Lat": 30.6184, "Lon": -96.3456, "Alt": 111.0},
    4079: {"Lat": 34.0008, "Lon": -81.0352, "Alt": 89.0},
    4080: {"Lat": 39.9623, "Lon": -83.0007, "Alt": 231.0},
    4081: {"Lat": 35.0892, "Lon": -92.4398, "Alt": 97.0},
    4082: {"Lat": 33.836, "Lon": -79.0478, "Alt": 15.0},
    4083: {"Lat": 36.1624, "Lon": -85.4997, "Alt": 345.0},
    4084: {"Lat": 25.7331, "Lon": -80.2585, "Alt": 11.0},
    4085: {"Lat": 27.7635, "Lon": -97.4033, "Alt": 10.0},
    4086: {"Lat": 44.5646, "Lon": -123.262, "Alt": 73.0},
    4087: {"Lat": 35.3138, "Lon": -83.1766, "Alt": 642.0},
    4088: {"Lat": 32.7763, "Lon": -96.7969, "Alt": 133.0},
    4089: {"Lat": 35.7902, "Lon": -80.2115, "Alt": 240.0},
    4090: {"Lat": 38.5436, "Lon": -121.739, "Alt": 18.0},
    4091: {"Lat": 39.7589, "Lon": -84.1916, "Alt": 228.0},
    4092: {"Lat": 29.2108, "Lon": -81.0228, "Alt": 4.0},
    4093: {"Lat": 41.8903, "Lon": -88.7714, "Alt": 261.0},
    4094: {"Lat": 29.0281, "Lon": -81.3034, "Alt": 15.0},
    4095: {"Lat": 33.1839, "Lon": -97.1413, "Alt": 194.0},
    4096: {"Lat": 39.7392, "Lon": -104.9849, "Alt": 1609.0},
    4097: {"Lat": 41.5869, "Lon": -93.6249, "Alt": 250.0},
    4098: {"Lat": 42.3316, "Lon": -83.0466, "Alt": 183.0},
    4099: {"Lat": 39.1582, "Lon": -75.5244, "Alt": 12.0},
    4100: {"Lat": 34.0029, "Lon": -84.1441, "Alt": 336.0},
    4101: {"Lat": 35.9967, "Lon": -78.9018, "Alt": 127.0},
    4102: {"Lat": 43.1346, "Lon": -70.927, "Alt": 19.0},
    4103: {"Lat": 42.732, "Lon": -84.4722, "Alt": 263.0},
    4104: {"Lat": 40.834, "Lon": -74.0971, "Alt": 26.0},
    4105: {"Lat": 40.6916, "Lon": -75.21, "Alt": 69.0},
    4106: {"Lat": 26.3014, "Lon": -98.1625, "Alt": 32.0},
    4107: {"Lat": 38.8114, "Lon": -89.9532, "Alt": 168.0},
    4108: {"Lat": 31.7601, "Lon": -106.487, "Alt": 1141.0},
    4109: {"Lat": 36.1029, "Lon": -79.5067, "Alt": 214.0},
    4110: {"Lat": 39.7045, "Lon": -77.3269, "Alt": 136.0},
    4111: {"Lat": 26.4381, "Lon": -81.8068, "Alt": 4.0},
    4112: {"Lat": 44.0505, "Lon": -123.0951, "Alt": 130.0},
    4113: {"Lat": 42.047, "Lon": -87.6846, "Alt": 185.0},
    4114: {"Lat": 37.9705, "Lon": -87.5716, "Alt": 123.0},
    4115: {"Lat": 38.8462, "Lon": -77.3064, "Alt": 134.0},
    4116: {"Lat": 41.1412, "Lon": -73.2637, "Alt": 8.0},
    4117: {"Lat": 46.8772, "Lon": -96.7898, "Alt": 277.0},
    4118: {"Lat": 37.3025, "Lon": -78.3924, "Alt": 100.0},
    4119: {"Lat": 36.0626, "Lon": -94.1574, "Alt": 434.0},
    4120: {"Lat": 35.1988, "Lon": -111.6518, "Alt": 2106.0},
    4121: {"Lat": 40.5872, "Lon": -105.077, "Alt": 1524.0},
    4122: {"Lat": 31.2013, "Lon": -97.7084, "Alt": 226.0},
    4123: {"Lat": 26.6406, "Lon": -81.8723, "Alt": 5.0},
    4124: {"Lat": 41.08, "Lon": -85.1386, "Alt": 232.0},
    4125: {"Lat": 32.7532, "Lon": -97.3327, "Alt": 195.0},
    4126: {"Lat": 36.7394, "Lon": -119.7848, "Alt": 94.0},
    4127: {"Lat": 33.8708, "Lon": -117.9294, "Alt": 52.0},
    4128: {"Lat": 29.652, "Lon": -82.325, "Alt": 59.0},
    4129: {"Lat": 32.9126, "Lon": -96.6389, "Alt": 169.0},
    4130: {"Lat": 33.5387, "Lon": -112.186, "Alt": 350.0},
    4131: {"Lat": 43.3099, "Lon": -73.6444, "Alt": 110.0},
    4132: {"Lat": 32.5277, "Lon": -92.714, "Alt": 93.0},
    4133: {"Lat": 47.9252, "Lon": -97.0306, "Alt": 257.0},
    4134: {"Lat": 40.4233, "Lon": -104.7091, "Alt": 1426.0},
    4135: {"Lat": 44.5126, "Lon": -88.0126, "Alt": 179.0},
    4136: {"Lat": 36.0726, "Lon": -79.792, "Alt": 257.0},
    4137: {"Lat": 33.4111, "Lon": -91.0636, "Alt": 41.0},
    4138: {"Lat": 35.6132, "Lon": -77.3725, "Alt": 22.0},
    4139: {"Lat": 34.8514, "Lon": -82.3985, "Alt": 300.0},
    4140: {"Lat": 33.5162, "Lon": -90.1795, "Alt": 41.0},
    4141: {"Lat": 30.3674, "Lon": -89.0928, "Alt": 8.0},
    4142: {"Lat": 40.8871, "Lon": -74.0411, "Alt": 8.0},
    4143: {"Lat": 41.3836, "Lon": -72.902, "Alt": 24.0},
    4144: {"Lat": 43.6308, "Lon": -74.4659, "Alt": 917.0},
    4145: {"Lat": 30.5046, "Lon": -90.4759, "Alt": 14.0},
    4146: {"Lat": 37.0264, "Lon": -76.3443, "Alt": 5.0},
    4147: {"Lat": 43.7024, "Lon": -72.2892, "Alt": 167.0},
    4148: {"Lat": 38.4493, "Lon": -78.8689, "Alt": 408.0},
    4149: {"Lat": 41.7646, "Lon": -72.6909, "Alt": 20.0},
    4150: {"Lat": 31.3271, "Lon": -89.2903, "Alt": 53.0},
    4151: {"Lat": 40.6737, "Lon": -73.6191, "Alt": 15.0},
    4152: {"Lat": 35.9557, "Lon": -80.0053, "Alt": 288.0},
    4153: {"Lat": 39.0331, "Lon": -84.4522, "Alt": 259.0},
    4154: {"Lat": 42.0427, "Lon": -88.0793, "Alt": 238.0},
    4155: {"Lat": 21.3045, "Lon": -157.8557, "Alt": 4.0},
    4156: {"Lat": 34.5038, "Lon": -93.0552, "Alt": 0.0},
    4157: {"Lat": 29.7589, "Lon": -95.3677, "Alt": 21.0},
    4158: {"Lat": 38.4192, "Lon": -82.4452, "Alt": 177.0},
    4159: {"Lat": 34.7298, "Lon": -86.5859, "Alt": 191.0},
    4160: {"Lat": 30.7235, "Lon": -95.5508, "Alt": 113.0},
    4161: {"Lat": 39.7683, "Lon": -86.1584, "Alt": 219.0},
    4162: {"Lat": 41.6613, "Lon": -91.5299, "Alt": 206.0},
    4163: {"Lat": 33.6857, "Lon": -117.826, "Alt": 16.0},
    4164: {"Lat": 42.4374, "Lon": -76.5484, "Alt": 305.0},
    4165: {"Lat": 33.4951, "Lon": -90.3229, "Alt": 38.0},
    4166: {"Lat": 32.2999, "Lon": -90.183, "Alt": 94.0},
    4167: {"Lat": 33.8138, "Lon": -85.7613, "Alt": 218.0},
    4168: {"Lat": 30.3262, "Lon": -81.6579, "Alt": 4.0},
    4169: {"Lat": 40.7216, "Lon": -74.0475, "Alt": 7.0},
    4170: {"Lat": 36.3168, "Lon": -82.3529, "Alt": 496.0},
    4171: {"Lat": 40.3267, "Lon": -78.922, "Alt": 358.0},
    4172: {"Lat": 35.8348, "Lon": -90.7045, "Alt": 106.0},
    4173: {"Lat": 42.2917, "Lon": -85.5872, "Alt": 239.0},
    4174: {"Lat": 39.1001, "Lon": -94.5781, "Alt": 281.0},
    4175: {"Lat": 29.7858, "Lon": -95.8244, "Alt": 42.0},
    4176: {"Lat": 34.0234, "Lon": -84.6155, "Alt": 336.0},
    4177: {"Lat": 41.1513, "Lon": -81.3578, "Alt": 323.0},
    4178: {"Lat": 47.3827, "Lon": -122.227, "Alt": 13.0},
    4179: {"Lat": 41.4729, "Lon": -71.5223, "Alt": 81.0},
    4180: {"Lat": 35.9604, "Lon": -83.921, "Alt": 276.0},
    4181: {"Lat": 30.2262, "Lon": -92.0178, "Alt": 14.0},
    4182: {"Lat": 20.8872, "Lon": -156.6748, "Alt": 46.0},
    4183: {"Lat": 28.3717, "Lon": -81.524, "Alt": 30.0},
    4184: {"Lat": 30.2305, "Lon": -93.217, "Alt": 3.0},
    4185: {"Lat": 28.0395, "Lon": -81.9498, "Alt": 63.0},
    4186: {"Lat": 41.3116, "Lon": -105.5918, "Alt": 2184.0},
    4187: {"Lat": 32.314, "Lon": -106.7798, "Alt": 1190.0},
    4188: {"Lat": 36.1674, "Lon": -115.1484, "Alt": 620.0},
    4189: {"Lat": 38.9719, "Lon": -95.2359, "Alt": 263.0},
    4190: {"Lat": 40.301, "Lon": -74.7374, "Alt": 59.0},
    4191: {"Lat": 40.9645, "Lon": -76.8848, "Alt": 143.0},
    4192: {"Lat": 43.1726, "Lon": -79.0359, "Alt": 113.0},
    4193: {"Lat": 38.0464, "Lon": -84.497, "Alt": 290.0},
    4194: {"Lat": 37.784, "Lon": -79.4428, "Alt": 324.0},
    4195: {"Lat": 40.8089, "Lon": -96.7078, "Alt": 360.0},
    4196: {"Lat": 34.7465, "Lon": -92.2896, "Alt": 102.0},
    4197: {"Lat": 41.7313, "Lon": -111.8349, "Alt": 1386.0},
    4198: {"Lat": 33.769, "Lon": -118.1916, "Alt": 17.0},
    4199: {"Lat": 40.5029, "Lon": -78.6314, "Alt": 595.0},
    4200: {"Lat": 31.8204, "Lon": -91.0501, "Alt": 72.0},
    4201: {"Lat": 34.0537, "Lon": -118.2428, "Alt": 96.0},
    4202: {"Lat": 42.7032, "Lon": -73.7636, "Alt": 122.0},
    4203: {"Lat": 38.2542, "Lon": -85.7594, "Alt": 144.0},
    4204: {"Lat": 42.6414, "Lon": -71.3085, "Alt": 31.0},
    4205: {"Lat": 33.5856, "Lon": -101.847, "Alt": 977.0},
    4206: {"Lat": 37.4138, "Lon": -79.1422, "Alt": 201.0},
    4207: {"Lat": 40.4589, "Lon": -90.6714, "Alt": 218.0},
    4208: {"Lat": 32.8407, "Lon": -83.6324, "Alt": 119.0},
    4209: {"Lat": 43.0747, "Lon": -89.3842, "Alt": 295.0},
    4210: {"Lat": 34.0356, "Lon": -118.6894, "Alt": 5.0},
    4211: {"Lat": 39.1836, "Lon": -96.5717, "Alt": 315.0},
    4212: {"Lat": 36.3433, "Lon": -88.8504, "Alt": 126.0},
    4213: {"Lat": 35.146, "Lon": -90.0518, "Alt": 86.0},
    4214: {"Lat": 25.7742, "Lon": -80.1936, "Alt": 7.0},
    4215: {"Lat": 43.0386, "Lon": -87.9091, "Alt": 178.0},
    4216: {"Lat": 44.9773, "Lon": -93.2655, "Alt": 267.0},
    4217: {"Lat": 46.8701, "Lon": -113.9953, "Alt": 980.0},
    4218: {"Lat": 30.6913, "Lon": -88.0438, "Alt": 6.0},
    4219: {"Lat": 32.5025, "Lon": -92.1162, "Alt": 26.0},
    4220: {"Lat": 32.3777, "Lon": -86.3091, "Alt": 64.0},
    4221: {"Lat": 40.5061, "Lon": -80.2082, "Alt": 361.0},
    4222: {"Lat": 37.8349, "Lon": -122.1288, "Alt": 150.0},
    4223: {"Lat": 38.184, "Lon": -83.4327, "Alt": 225.0},
    4224: {"Lat": 39.6297, "Lon": -79.9559, "Alt": 271.0},
    4225: {"Lat": 46.7324, "Lon": -117.0002, "Alt": 789.0},
    4226: {"Lat": 43.5976, "Lon": -84.7668, "Alt": 236.0},
    4227: {"Lat": 40.1937, "Lon": -85.3865, "Alt": 291.0},
    4228: {"Lat": 35.846, "Lon": -86.3921, "Alt": 192.0},
    4229: {"Lat": 36.6106, "Lon": -88.3021, "Alt": 159.0},
    4230: {"Lat": 33.6956, "Lon": -78.89, "Alt": 8.0},
    4231: {"Lat": 31.5971, "Lon": -94.5927, "Alt": 109.0},
    4232: {"Lat": 43.578, "Lon": -116.5617, "Alt": 759.0},
    4233: {"Lat": 36.1623, "Lon": -86.7743, "Alt": 137.0},
    4234: {"Lat": 31.7607, "Lon": -93.086, "Alt": 39.0},
    4235: {"Lat": 41.6612, "Lon": -72.7795, "Alt": 55.0},
    4236: {"Lat": 41.3082, "Lon": -72.9251, "Alt": 14.0},
    4237: {"Lat": 29.9561, "Lon": -90.0734, "Alt": 3.0},
    4238: {"Lat": 40.9115, "Lon": -73.7826, "Alt": 28.0},
    4239: {"Lat": 40.7127, "Lon": -74.006, "Alt": 32.0},
    4240: {"Lat": 39.6828, "Lon": -75.7516, "Alt": 43.0},
    4241: {"Lat": 40.7357, "Lon": -74.1724, "Alt": 14.0},
    4242: {"Lat": 43.0848, "Lon": -79.0702, "Alt": 99.0},
    4243: {"Lat": 30.5169, "Lon": -86.4822, "Alt": 17.0},
    4244: {"Lat": 36.8494, "Lon": -76.29, "Alt": 7.0},
    4245: {"Lat": 34.789, "Lon": -86.5719, "Alt": 219.0},
    4246: {"Lat": 40.5093, "Lon": -88.9844, "Alt": 239.0},
    4247: {"Lat": 35.2226, "Lon": -97.4395, "Alt": 358.0},
    4248: {"Lat": 34.7695, "Lon": -92.2671, "Alt": 85.0},
    4249: {"Lat": 34.2346, "Lon": -118.5369, "Alt": 257.0},
    4250: {"Lat": 41.7046, "Lon": -86.2382, "Alt": 225.0},
    4251: {"Lat": 37.8045, "Lon": -122.2714, "Alt": 9.0},
    4252: {"Lat": 38.3387, "Lon": -87.345, "Alt": 141.0},
    4253: {"Lat": 41.223, "Lon": -111.9738, "Alt": 1316.0},
    4254: {"Lat": 35.473, "Lon": -97.5171, "Alt": 372.0},
    4255: {"Lat": 41.2587, "Lon": -95.9384, "Alt": 330.0},
    4256: {"Lat": 33.4055, "Lon": -80.7784, "Alt": 52.0},
    4257: {"Lat": 40.2982, "Lon": -111.6944, "Alt": 1458.0},
    4258: {"Lat": 28.5421, "Lon": -81.379, "Alt": 35.0},
    4259: {"Lat": 44.8836, "Lon": -68.6728, "Alt": 39.0},
    4260: {"Lat": 34.3664, "Lon": -89.5188, "Alt": 159.0},
    4261: {"Lat": 39.5103, "Lon": -84.7421, "Alt": 293.0},
    4262: {"Lat": 36.0724, "Lon": -115.1475, "Alt": 641.0},
    4263: {"Lat": 33.8495, "Lon": -78.6616, "Alt": 1.0},
    4264: {"Lat": 21.3522, "Lon": -157.9552, "Alt": 4.0},
    4265: {"Lat": 40.6939, "Lon": -89.5891, "Alt": 153.0},
    4266: {"Lat": 39.9527, "Lon": -75.1635, "Alt": 12.0},
    4267: {"Lat": 33.4484, "Lon": -112.0741, "Alt": 333.0},
    4268: {"Lat": 34.2229, "Lon": -92.0043, "Alt": 71.0},
    4269: {"Lat": 40.5464, "Lon": -74.4661, "Alt": 26.0},
    4270: {"Lat": 40.4407, "Lon": -80.0026, "Alt": 237.0},
    4271: {"Lat": 20.6461, "Lon": -87.0809, "Alt": 8.0},
    4272: {"Lat": 42.862, "Lon": -112.4506, "Alt": 1364.0},
    4273: {"Lat": 43.6574, "Lon": -70.2587, "Alt": 27.0},
    4274: {"Lat": 45.5202, "Lon": -122.6742, "Alt": 19.0},
    4275: {"Lat": 41.7066, "Lon": -73.9284, "Alt": 51.0},
    4276: {"Lat": 30.0858, "Lon": -95.9907, "Alt": 77.0},
    4277: {"Lat": 34.004, "Lon": -81.0306, "Alt": 104.0},
    4278: {"Lat": 34.5951, "Lon": -112.3339, "Alt": 1559.0},
    4279: {"Lat": 38.2028, "Lon": -75.692, "Alt": 11.0},
    4280: {"Lat": 40.3497, "Lon": -74.6597, "Alt": 66.0},
    4281: {"Lat": 41.824, "Lon": -71.4128, "Alt": 10.0},
    4282: {"Lat": 40.2337, "Lon": -111.6587, "Alt": 1392.0},
    4283: {"Lat": 20.6407, "Lon": -105.2203, "Alt": 0.0},
    4284: {"Lat": 46.7296, "Lon": -117.1799, "Alt": 717.0},
    4285: {"Lat": 40.7135, "Lon": -73.8283, "Alt": 30.0},
    4286: {"Lat": 37.1343, "Lon": -80.5749, "Alt": 553.0},
    4287: {"Lat": 35.7804, "Lon": -78.6391, "Alt": 113.0},
    4288: {"Lat": 49.4607, "Lon": 7.5563, "Alt": 256.0},
    4289: {"Lat": 44.0806, "Lon": -103.228, "Alt": 990.0},
    4290: {"Lat": 39.5262, "Lon": -119.8127, "Alt": 1379.0},
    4291: {"Lat": 37.7479, "Lon": -84.2947, "Alt": 293.0},
    4292: {"Lat": 37.5385, "Lon": -77.4343, "Alt": 54.0},
    4293: {"Lat": 40.9006, "Lon": -73.9064, "Alt": 67.0},
    4294: {"Lat": 33.9825, "Lon": -117.3742, "Alt": 262.0},
    4295: {"Lat": 42.6806, "Lon": -83.1338, "Alt": 232.0},
    4296: {"Lat": 43.1573, "Lon": -77.6152, "Alt": 155.0},
    4297: {"Lat": 34.9249, "Lon": -81.0251, "Alt": 210.0},
    4298: {"Lat": 41.9941, "Lon": -87.8757, "Alt": 195.0},
    4299: {"Lat": 32.5297, "Lon": -92.6387, "Alt": 99.0},
    4300: {"Lat": 38.5811, "Lon": -121.4939, "Alt": 16.0},
    4301: {"Lat": 40.7596, "Lon": -111.8868, "Alt": 1305.0},
    4302: {"Lat": 29.4246, "Lon": -98.4951, "Alt": 201.0},
    4303: {"Lat": 32.7174, "Lon": -117.1628, "Alt": 29.0},
    4304: {"Lat": 37.7879, "Lon": -122.4075, "Alt": 28.0},
    4305: {"Lat": 37.3362, "Lon": -121.8906, "Alt": 34.0},
    4307: {"Lat": 18.4653, "Lon": -66.1167, "Alt": 33.0},
    4308: {"Lat": 35.354, "Lon": -120.3757, "Alt": 529.0},
    4309: {"Lat": 29.8826, "Lon": -97.9406, "Alt": 193.0},
    4310: {"Lat": 34.4221, "Lon": -119.7027, "Alt": 28.0},
    4311: {"Lat": 37.3541, "Lon": -121.9552, "Alt": 23.0},
    4312: {"Lat": 32.079, "Lon": -81.0921, "Alt": 17.0},
    4313: {"Lat": 47.6038, "Lon": -122.3301, "Alt": 40.0},
    4314: {"Lat": 41.896, "Lon": -87.625, "Alt": 191.0},
    4315: {"Lat": 32.5135, "Lon": -93.7478, "Alt": 64.0},
    4316: {"Lat": 43.5476, "Lon": -96.7294, "Alt": 435.0},
    4317: {"Lat": 41.922, "Lon": -71.5495, "Alt": 81.0},
    4318: {"Lat": 41.6834, "Lon": -86.25, "Alt": 216.0},
    4319: {"Lat": 36.436, "Lon": 126.9428, "Alt": 102.0},
    4320: {"Lat": 40.7475, "Lon": -74.2635, "Alt": 79.0},
    4321: {"Lat": 26.1037, "Lon": -97.1647, "Alt": 4.0},
    4322: {"Lat": 34.9874, "Lon": -90.0035, "Alt": 111.0},
    4323: {"Lat": 34.9498, "Lon": -81.932, "Alt": 248.0},
    4324: {"Lat": 47.6572, "Lon": -117.4235, "Alt": 578.0},
    4325: {"Lat": 39.799, "Lon": -89.644, "Alt": 184.0},
    4326: {"Lat": 42.1019, "Lon": -72.5887, "Alt": 30.0},
    4327: {"Lat": 37.2082, "Lon": -93.2923, "Alt": 403.0},
    4328: {"Lat": 42.0805, "Lon": -78.4772, "Alt": 433.0},
    4329: {"Lat": 38.6254, "Lon": -90.19, "Alt": 146.0},
    4330: {"Lat": 18.3429, "Lon": -64.9189, "Alt": 53.0},
    4331: {"Lat": 37.4265, "Lon": -122.1703, "Alt": 33.0},
    4332: {"Lat": 33.4639, "Lon": -88.8152, "Alt": 119.0},
    4333: {"Lat": 40.5835, "Lon": -74.1496, "Alt": 50.0},
    4334: {"Lat": 32.449, "Lon": -81.7833, "Alt": 82.0},
    4335: {"Lat": 36.1156, "Lon": -97.0586, "Alt": 278.0},
    4336: {"Lat": 37.9577, "Lon": -121.2908, "Alt": 5.0},
    4337: {"Lat": 40.9085, "Lon": -73.1325, "Alt": 53.0},
    4338: {"Lat": 41.8086, "Lon": -72.2513, "Alt": 199.0},
    4339: {"Lat": 26.1667, "Lon": -80.2788, "Alt": 5.0},
    4340: {"Lat": 43.0481, "Lon": -76.1474, "Alt": 126.0},
    4341: {"Lat": 30.4381, "Lon": -84.2809, "Alt": 72.0},
    4342: {"Lat": 27.945, "Lon": -82.4583, "Alt": 1.0},
    4343: {"Lat": 33.4255, "Lon": -111.94, "Alt": 357.0},
    4344: {"Lat": 39.4667, "Lon": -87.4139, "Alt": 150.0},
    4345: {"Lat": 29.7958, "Lon": -90.8229, "Alt": 5.0},
    4346: {"Lat": 41.6529, "Lon": -83.5378, "Alt": 179.0},
    4347: {"Lat": 18.3351, "Lon": -64.9105, "Alt": 66.0},
    4348: {"Lat": 39.3965, "Lon": -76.6148, "Alt": 115.0},
    4349: {"Lat": 31.8088, "Lon": -85.97, "Alt": 166.0},
    4350: {"Lat": 32.2229, "Lon": -110.9748, "Alt": 722.0},
    4351: {"Lat": 36.1563, "Lon": -95.9928, "Alt": 217.0},
    4352: {"Lat": 34.2576, "Lon": -88.7034, "Alt": 87.0},
    4353: {"Lat": 33.2096, "Lon": -87.5675, "Alt": 70.0},
    4354: {"Lat": 41.4347, "Lon": -72.11, "Alt": 29.0},
    4355: {"Lat": 40.8129, "Lon": -77.8711, "Alt": 355.0},
    4356: {"Lat": 38.9788, "Lon": -104.8644, "Alt": 2107.0},
    4357: {"Lat": 41.4673, "Lon": -87.0604, "Alt": 239.0},
    4358: {"Lat": 45.6307, "Lon": -122.6745, "Alt": 30.0},
    4359: {"Lat": 42.7795, "Lon": -96.9291, "Alt": 375.0},
    4360: {"Lat": 42.0851, "Lon": -76.0538, "Alt": 251.0},
    4361: {"Lat": 40.0373, "Lon": -75.3491, "Alt": 121.0},
    4362: {"Lat": 31.5492, "Lon": -97.1475, "Alt": 148.0},
    4363: {"Lat": 38.895, "Lon": -77.0365, "Alt": 12.0},
    4364: {"Lat": 41.762, "Lon": -72.742, "Alt": 43.0},
    4365: {"Lat": 40.4259, "Lon": -86.9081, "Alt": 189.0},
    4366: {"Lat": 40.2904, "Lon": -74.0173, "Alt": 9.0},
    4367: {"Lat": 41.3926, "Lon": -73.9599, "Alt": 57.0},
    4368: {"Lat": 37.6922, "Lon": -97.3375, "Alt": 398.0},
    4369: {"Lat": 37.2709, "Lon": -76.7074, "Alt": 30.0},
    4370: {"Lat": 34.2353, "Lon": -77.9487, "Alt": 12.0},
    4371: {"Lat": 36.0998, "Lon": -80.2441, "Alt": 282.0},
    4372: {"Lat": 42.2626, "Lon": -71.8019, "Alt": 153.0},
    4373: {"Lat": 41.1036, "Lon": -80.652, "Alt": 272.0},
    4374: {"Lat": 42.2411, "Lon": -83.6118, "Alt": 223.0},
    4376: {"Lat": 30.4008, "Lon": -88.8894, "Alt": 6.0},
    4382: {"Lat": 32.3479, "Lon": -86.0741, "Alt": 67.0},
    4383: {"Lat": 18.3592, "Lon": -66.1157, "Alt": 27.0},
    4385: {"Lat": 21.6435, "Lon": -157.9282, "Alt": 2.0},
    4387: {"Lat": 41.5058, "Lon": -90.5137, "Alt": 180.0},
    4389: {"Lat": 40.7127, "Lon": -74.006, "Alt": 32.0},
    4390: {"Lat": 32.8546, "Lon": -79.9748, "Alt": 7.0},
    4391: {"Lat": 29.1872, "Lon": -82.1401, "Alt": 24.0},
    4392: {"Lat": 30.1587, "Lon": -85.6603, "Alt": 11.0},
    4396: {"Lat": 38.7878, "Lon": -90.6747, "Alt": 0.0},
    4399: {"Lat": 18.3371, "Lon": -66.0017, "Alt": 56.0},
    4402: {"Lat": 38.8162, "Lon": -76.7517, "Alt": 13.0},
    4403: {"Lat": 20.8905, "Lon": -156.5031, "Alt": 80.0},
    4404: {"Lat": 28.5978, "Lon": -81.351, "Alt": 30.0},
    4406: {"Lat": 40.7266, "Lon": -73.6343, "Alt": 29.0},
    4407: {"Lat": 47.5049, "Lon": -111.2919, "Alt": 1018.0},
    4408: {"Lat": 40.3353, "Lon": -75.9279, "Alt": 84.0},
    4409: {"Lat": 40.7158, "Lon": -73.5899, "Alt": 24.0},
    4410: {"Lat": 44.4255, "Lon": -69.0074, "Alt": 33.0},
    4412: {"Lat": 38.4393, "Lon": -75.0582, "Alt": 2.0},
    4413: {"Lat": 30.7069, "Lon": -88.1712, "Alt": 40.0},
    4414: {"Lat": 33.1506, "Lon": -96.8238, "Alt": 211.0},
    4415: {"Lat": 40.1164, "Lon": -82.9946, "Alt": 282.0},
    4416: {"Lat": 25.8632, "Lon": -80.1928, "Alt": 4.0},
    4417: {"Lat": 43.6156, "Lon": -84.2472, "Alt": 193.0},
    4421: {"Lat": 31.3119, "Lon": -92.4454, "Alt": 26.0},
    4422: {"Lat": 34.6713, "Lon": -86.7336, "Alt": 195.0},
    4423: {"Lat": 40.4798, "Lon": -88.9939, "Alt": 252.0},
    4424: {"Lat": 18.3641, "Lon": -65.9527, "Alt": 27.0},
    4425: {"Lat": 34.2006, "Lon": -90.5702, "Alt": 55.0},
    4426: {"Lat": 37.3229, "Lon": -122.0323, "Alt": 74.0},
    4427: {"Lat": 26.0629, "Lon": -80.2331, "Alt": 2.0},
    4428: {"Lat": 40.6661, "Lon": -89.5801, "Alt": 142.0},
    4429: {"Lat": 26.1223, "Lon": -80.1434, "Alt": 4.0},
    4430: {"Lat": 35.0074, "Lon": -80.9451, "Alt": 200.0},
    4431: {"Lat": 42.9632, "Lon": -85.6679, "Alt": 203.0},
    4432: {"Lat": 19.7074, "Lon": -155.0816, "Alt": 29.0},
    4433: {"Lat": 29.5958, "Lon": -90.7195, "Alt": 4.0},
    4434: {"Lat": 30.2224, "Lon": -92.6571, "Alt": 10.0},
    4435: {"Lat": 46.2087, "Lon": -119.1199, "Alt": 114.0},
    4436: {"Lat": 42.3684, "Lon": -83.3527, "Alt": 195.0},
    4437: {"Lat": 26.5293, "Lon": -81.9251, "Alt": 3.0},
    4438: {"Lat": 28.0785, "Lon": -80.6078, "Alt": 8.0},
    4439: {"Lat": 33.7584, "Lon": -89.9568, "Alt": 65.0},
    4440: {"Lat": 32.7941, "Lon": -79.8626, "Alt": 8.0},
    4441: {"Lat": 43.1366, "Lon": -79.0349, "Alt": 182.0},
    4442: {"Lat": 39.7595, "Lon": -87.5011, "Alt": 194.0},
    4443: {"Lat": 26.7154, "Lon": -80.0533, "Alt": 6.0},
    4444: {"Lat": 34.7998, "Lon": -87.6773, "Alt": 171.0},
    4445: {"Lat": 38.21, "Lon": -84.5597, "Alt": 259.0},
    4446: {"Lat": 36.6259, "Lon": -121.8169, "Alt": 95.0},
    4447: {"Lat": 28.2919, "Lon": -81.4076, "Alt": 23.0},
    4448: {"Lat": 42.6619, "Lon": -83.3804, "Alt": 295.0},
    4449: {"Lat": 27.7712, "Lon": -82.634, "Alt": 5.0},
    4450: {"Lat": 37.5687, "Lon": -84.2963, "Alt": 315.0},
    4451: {"Lat": 32.3063, "Lon": -92.4504, "Alt": 56.0},
    4452: {"Lat": 20.8748, "Lon": -156.453, "Alt": 14.0},
    4453: {"Lat": 43.8123, "Lon": -91.2514, "Alt": 206.0},
    4454: {"Lat": 20.803, "Lon": -156.3107, "Alt": 807.0},
    4455: {"Lat": 40.7987, "Lon": -74.239, "Alt": 146.0},
    4456: {"Lat": 41.034, "Lon": -73.7629, "Alt": 72.0},
    4457: {"Lat": 28.5657, "Lon": -81.5857, "Alt": 41.0},
    4458: {"Lat": 33.4737, "Lon": -86.8085, "Alt": 217.0},
    4459: {"Lat": 34.4146, "Lon": -119.8569, "Alt": 15.0},
    4460: {"Lat": 41.7778, "Lon": -72.2111, "Alt": 120.0},
    4461: {"Lat": 41.4488, "Lon": -72.1295, "Alt": 54.0},
    4462: {"Lat": 42.703, "Lon": -71.1309, "Alt": 27.0},
    4463: {"Lat": 38.9519, "Lon": -92.3337, "Alt": 216.0},
    4464: {"Lat": 40.2877, "Lon": -74.7123, "Alt": 28.0},
    4465: {"Lat": 42.0822, "Lon": -78.4294, "Alt": 436.0},
    4466: {"Lat": 39.8209, "Lon": -84.0194, "Alt": 256.0},
    4467: {"Lat": 40.0238, "Lon": -75.3173, "Alt": 126.0},
    4468: {"Lat": 40.5061, "Lon": -80.2082, "Alt": 361.0},
    4469: {"Lat": 41.4549, "Lon": -71.5395, "Alt": 37.0},
    4470: {"Lat": 39.9234, "Lon": -83.8101, "Alt": 298.0},
    4471: {"Lat": 31.8369, "Lon": -102.0104, "Alt": 830.0},
    4472: {"Lat": 20.6219, "Lon": -87.0757, "Alt": 10.0},
    4473: {"Lat": 39.3371, "Lon": -95.7213, "Alt": 362.0},
    4474: {"Lat": 28.7998, "Lon": -97.0064, "Alt": 33.0},
    4475: {"Lat": 39.7459, "Lon": -75.5466, "Alt": 33.0},
    4476: {"Lat": 37.1097, "Lon": -113.5613, "Alt": 862.0},
    4477: {"Lat": 32.2201, "Lon": -98.2045, "Alt": 393.0},
    4478: {"Lat": 32.8459, "Lon": -117.2576, "Alt": 81.0},
    4479: {"Lat": 31.868, "Lon": -91.1334, "Alt": 83.0},
    4480: {"Lat": 40.7144, "Lon": -73.7179, "Alt": 22.0},
    4481: {"Lat": 33.5017, "Lon": -117.6626, "Alt": 37.0},
    4482: {"Lat": 44.9497, "Lon": -93.0931, "Alt": 237.0},
    4483: {"Lat": 34.979, "Lon": -101.9282, "Alt": 1085.0},
    4484: {"Lat": 26.1422, "Lon": -81.7943, "Alt": 3.0},
    4485: {"Lat": 36.9744, "Lon": -122.0295, "Alt": 9.0},
    4486: {"Lat": 36.032, "Lon": -114.9823, "Alt": 595.0},
    4487: {"Lat": 30.4213, "Lon": -87.2169, "Alt": 20.0},
    4488: {"Lat": 33.2436, "Lon": -95.9027, "Alt": 174.0},
    4489: {"Lat": 32.8236, "Lon": -96.7699, "Alt": 179.0},
    4490: {"Lat": 42.0245, "Lon": -71.1287, "Alt": 41.0},
    4491: {"Lat": 39.6556, "Lon": -78.8106, "Alt": 240.0},
    4492: {"Lat": 37.1283, "Lon": -84.0836, "Alt": 381.0},
    4493: {"Lat": 40.6652, "Lon": -74.3045, "Alt": 23.0},
    4494: {"Lat": 20.7476, "Lon": -156.455, "Alt": 5.0},
    4495: {"Lat": 43.0387, "Lon": -76.073, "Alt": 175.0},
    4496: {"Lat": 47.4254, "Lon": -121.7768, "Alt": 280.0},
    4497: {"Lat": 33.7288, "Lon": -116.3826, "Alt": 56.0},
    4498: {"Lat": 29.8947, "Lon": -81.3145, "Alt": 4.0},
    4499: {"Lat": 40.2203, "Lon": -74.7659, "Alt": 23.0},
    4500: {"Lat": 35.7333, "Lon": -81.3443, "Alt": 354.0},
    4501: {"Lat": 30.3935, "Lon": -86.4958, "Alt": 10.0},
    4502: {"Lat": 42.0473, "Lon": -71.0814, "Alt": 35.0},
    4503: {"Lat": 35.1184, "Lon": -84.0886, "Alt": 533.0},
    4504: {"Lat": 39.7686, "Lon": -94.8466, "Alt": 274.0},
    4505: {"Lat": 48.844, "Lon": 2.359, "Alt": 38.0},
    4506: {"Lat": 37.666, "Lon": -77.5064, "Alt": 64.0},
    4507: {"Lat": 33.5801, "Lon": -85.0766, "Alt": 337.0},
    4508: {"Lat": 42.1295, "Lon": -80.0853, "Alt": 198.0},
    4509: {"Lat": 43.4888, "Lon": -112.0363, "Alt": 1438.0},
    4510: {"Lat": 33.9562, "Lon": -118.3531, "Alt": 40.0},
    4511: {"Lat": 40.8937, "Lon": -72.8963, "Alt": 30.0},
    4512: {"Lat": 37.7965, "Lon": -80.2976, "Alt": 564.0},
    4513: {"Lat": 41.2465, "Lon": -75.8817, "Alt": 175.0},
    4514: {"Lat": 39.6848, "Lon": -83.9297, "Alt": 287.0},
    4515: {"Lat": 33.8246, "Lon": -116.5403, "Alt": 140.0},
    4516: {"Lat": 40.7981, "Lon": -88.2967, "Alt": 204.0},
    4517: {"Lat": 20.8511, "Lon": -156.3237, "Alt": 464.0},
    4518: {"Lat": 18.4182, "Lon": -66.4918, "Alt": 14.0},
    4519: {"Lat": 25.942, "Lon": -80.2456, "Alt": 5.0},
    4520: {"Lat": 38.9668, "Lon": -119.9443, "Alt": 1903.0},
    4521: {"Lat": 29.906, "Lon": -90.1423, "Alt": 3.0},
    4522: {"Lat": 38.2009, "Lon": -84.8733, "Alt": 162.0},
    4523: {"Lat": 42.2714, "Lon": -89.094, "Alt": 224.0},
    4524: {"Lat": 35.042, "Lon": -89.6645, "Alt": 118.0},
    4525: {"Lat": 33.6534, "Lon": -84.4494, "Alt": 321.0},
    4526: {"Lat": 41.2707, "Lon": -72.947, "Alt": 7.0},
    4527: {"Lat": 38.834, "Lon": -104.8253, "Alt": 1834.0},
    4528: {"Lat": 40.285, "Lon": -76.6535, "Alt": 121.0},
    4529: {"Lat": 18.3379, "Lon": -64.959, "Alt": 16.0},
    4530: {"Lat": 20.5067, "Lon": -87.2301, "Alt": 7.0},
    4531: {"Lat": 31.8457, "Lon": -102.3677, "Alt": 886.0},
}


In [29]:
import pandas as pd
import numpy as np

# CITY_GEO_DICT = { 4001: {"Lat": 32.45, "Lon": -99.75, "Alt": 531.0}, ... } (이전에 생성한 딕셔너리를 위에 두세요)

# -------------------------------------------------------------------------
# 1. 지리 정보 기본 프레임 준비
# -------------------------------------------------------------------------
geo_info = pd.DataFrame.from_dict(CITY_GEO_DICT, orient='index').rename_axis('CityID').reset_index()
cities_df = pd.read_csv('../provided/Cities.csv')

# -------------------------------------------------------------------------
# 2. 남녀 공통 전처리 함수 정의
# -------------------------------------------------------------------------
def generate_geography_features(gender='M'):
    """
    gender: 'M' (남성부) 또는 'W' (여성부)
    """
    print(f"\n[{gender}] 부문 지리 및 누적 피로도 피처 생성")
    
    # 1. 파일 로드 (접두사 M 또는 W에 따라 파일명 동적 할당)
    regular_results = pd.read_csv(f'../provided/{gender}RegularSeasonCompactResults.csv')
    game_cities = pd.read_csv(f'../provided/{gender}GameCities.csv')
    
    # 2. 각 팀의 "홈 구장(캠퍼스)이 속한 도시" 찾기
    home_games = regular_results[regular_results['WLoc'] == 'H'].copy()
    home_games['HomeTeamID'] = home_games['WTeamID']

    home_cities = pd.merge(home_games[['Season', 'DayNum', 'HomeTeamID']], 
                           game_cities, 
                           left_on=['Season', 'DayNum', 'HomeTeamID'], 
                           right_on=['Season', 'DayNum', 'WTeamID'])

    # 팀별 최빈값(가장 많이 홈경기를 치른 도시) 추출 
    # (주의: pandas 버전에 따라 mode() 사용법이 다를 수 있어 agg로 안전하게 처리)
    def get_mode(x):
        return x.mode().iloc[0] if not x.mode().empty else np.nan
        
    team_home_city = home_cities.groupby('HomeTeamID')['CityID'].agg(get_mode).reset_index()
    team_home_city.columns = ['TeamID', 'HomeCityID']

    # 팀별 고유 홈 위도/경도/고도 매핑
    team_geo_base = pd.merge(team_home_city, geo_info, left_on='HomeCityID', right_on='CityID', how='left')
    team_geo_base = team_geo_base[['TeamID', 'Lat', 'Lon', 'Alt']].rename(columns={
        'Lat': 'Home_Lat', 'Lon': 'Home_Lon', 'Alt': 'Home_Altitude'
    })

    # 3. 막판 30일 (DayNum 103~132)의 원정 누적 피로도 / 고도 충격량
    late_games = regular_results[(regular_results['DayNum'] >= 103) & (regular_results['DayNum'] <= 132)].copy()
    games_geo = pd.merge(late_games, game_cities, on=['Season', 'DayNum', 'WTeamID', 'LTeamID'], how='left')
    games_geo = pd.merge(games_geo, geo_info, on='CityID', how='left')

    fatigue_records = []
    
    # 전체 시즌-팀을 순회하며 피로도 누적 로직 실행 (속도 최적화를 위해 팀별로 그룹핑)
    for season, season_df in games_geo.groupby('Season'):
        teams_in_season = set(season_df['WTeamID']).union(set(season_df['LTeamID']))
        
        for team in teams_in_season:
            # 해당 팀의 경기만 필터링 (시간순 정렬)
            team_games = season_df[(season_df['WTeamID'] == team) | (season_df['LTeamID'] == team)].sort_values('DayNum')
            
            # 홈 좌표가 없는 팀은 스킵 (생성 실패했거나 과거 데이터라 매핑이 안 된 경우)
            home_geo = team_geo_base[team_geo_base['TeamID'] == team]
            if home_geo.empty or pd.isna(home_geo['Home_Lat'].values[0]):
                continue
                
            h_lat, h_lon, h_alt = home_geo['Home_Lat'].values[0], home_geo['Home_Lon'].values[0], home_geo['Home_Altitude'].values[0]
            
            c_alt_fatig, c_lon_fatig, c_dist_fatig, h_alt_wins = 0, 0, 0, 0
            
            for _, row in team_games.iterrows():
                g_lat, g_lon, g_alt = row['Lat'], row['Lon'], row['Alt']
                if pd.isna(g_lat): continue
                    
                # 홈('H') 경기가 아닌 원정/중립 경기만 피로도로 누적
                is_home_game = (row['WTeamID'] == team and row['WLoc'] == 'H')
                if not is_home_game:
                    c_alt_fatig += abs(g_alt - h_alt)
                    c_lon_fatig += abs(g_lon - h_lon)
                    c_dist_fatig += np.sqrt((g_lat - h_lat)**2 + (g_lon - h_lon)**2)
                    
                    # 내 홈구장보다 1000m 높은 고산지대 원정에서 승리했는가?
                    if row['WTeamID'] == team and (g_alt - h_alt) >= 1000:
                        h_alt_wins += 1

            fatigue_records.append({
                'Season': season,
                'TeamID': team,
                'LateSeason_Altitude_Fatigue': round(c_alt_fatig, 1),
                'LateSeason_Lon_Fatigue': round(c_lon_fatig, 2),
                'LateSeason_Travel_Fatigue': round(c_dist_fatig, 2),
                'Season_High_Alt_Exp_Count': h_alt_wins
            })
            
    fatigue_df = pd.DataFrame(fatigue_records)

    # 4. 최종 데이터프레임 병합: (모든 팀-시즌 베이스) + (홈 좌표) + (피로도)
    base_season_teams = regular_results[['Season', 'WTeamID']].rename(columns={'WTeamID':'TeamID'})
    base_season_teams = pd.concat([base_season_teams, regular_results[['Season', 'LTeamID']].rename(columns={'LTeamID':'TeamID'})]).drop_duplicates()

    final_df = pd.merge(base_season_teams, team_geo_base, on='TeamID', how='left')
    if not fatigue_df.empty:
        final_df = pd.merge(final_df, fatigue_df, on=['Season', 'TeamID'], how='left')
        
    # 막판 30일에 원정을 뛰지 않은 팀들은 피로도를 0으로 채움
    fill_cols = ['LateSeason_Altitude_Fatigue', 'LateSeason_Lon_Fatigue', 'LateSeason_Travel_Fatigue', 'Season_High_Alt_Exp_Count']
    for col in fill_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna(0)

    # 5. CSV 저장
    output_filename = f"geography_travel_features_{gender}.csv"
    final_df.to_csv(output_filename, index=False)
    
    print(f"✅ [{gender}] 완료! 모양: {final_df.shape} -> 저장됨: {output_filename}")
    return final_df

# -------------------------------------------------------------------------
# 3. 함수 실행! (남/녀 모두 추출)
# -------------------------------------------------------------------------
geo_features_M = generate_geography_features('M')
geo_features_W = generate_geography_features('W')

display(geo_features_M.head(3))



[M] 부문 지리 및 누적 피로도 피처 생성
✅ [M] 완료! 모양: (13753, 9) -> 저장됨: geography_travel_features_M.csv

[W] 부문 지리 및 누적 피로도 피처 생성
✅ [W] 완료! 모양: (9851, 9) -> 저장됨: geography_travel_features_W.csv


,Season,TeamID,Home_Lat,Home_Lon,Home_Altitude,LateSeason_Altitude_Fatigue,LateSeason_Lon_Fatigue,LateSeason_Travel_Fatigue,Season_High_Alt_Exp_Count
0,1985,1228,40.1165,-88.2431,227.0,0.0,0.0,0.0,0.0
1,1985,1106,32.3777,-86.3091,64.0,0.0,0.0,0.0,0.0
2,1985,1112,32.2229,-110.9748,722.0,0.0,0.0,0.0,0.0
